# 2 - Self-supervised pretraining and niche discovery

Vignette 1 turned three MIBI-TOF images into 44 patch graphs. Here we:

1. pretrain a graph encoder with **BGRL**, which needs no labels,
2. attach a supervised head and export per-instance embeddings,
3. run the interpretability report to discover **spatial niches**.

---

### Read this before interpreting any number below

This dataset has **three images from two donors**. That is enough to
demonstrate the mechanics and enough for unsupervised niche discovery, but it
is *not* enough for supervised evaluation:

- Splitting by sample (the correct choice) puts one image in each of
  train/val/test.
- The only genuine bag-level variable is donor identity, and after that split
  the training image carries a single class.

So the supervised step here is **plumbing, not a result**. It exists because
`grass-mil-predict` needs a supervised checkpoint to export instance tables.
Treat the classifier as a feature extractor. The niche analysis at the end is
the part that stands on its own, because it is unsupervised.


## Setup


In [ ]:
import os
import pathlib
import subprocess
import sys
import glob

import pandas as pd

WORK = pathlib.Path("vignette_work").resolve()
RAW, PROCESSED = WORK / "data" / "raw", WORK / "data" / "processed"
assert PROCESSED.exists(), "Run vignette 1 first."
env = dict(os.environ, PROJECT_ROOT=str(WORK))

DATA_ARGS = [
    "data=spatial_omics",
    f"data.raw_manifest_path={RAW}/manifest.csv",
    f"data.processed_dir={PROCESSED}",
    "data.h5ad.coord_source=obsm",
    "data.h5ad.coord_key=spatial",
    "data.h5ad.categorical_label_columns=[cell_type]",
    "data.coord_scale_um=0.39",
    "data.sample_unit=tile",
    "data.tiling.tile_size_um=100",
    "data.tiling.stride_um=100",
    "data.min_cells=20",
    "data.tiling.min_cells=20",
    "data.split.train_val_test_split=[0.34,0.33,0.33]",
    "data.split.split_by=sample",
]


def run(args, tag):
    out = subprocess.run([sys.executable, "-m"] + args, capture_output=True, text=True, env=env)
    print(f"{tag}: exit {out.returncode}")
    if out.returncode:
        print(out.stdout[-2500:], out.stderr[-2500:])
        raise RuntimeError(f"{tag} failed - see output above")
    return out

## 1. BGRL pretraining (no labels)

BGRL learns node representations by matching two augmented views of the same
graph. Nothing here uses `donor_label`.

The augmentation preset matters. `bgrl_paper` adds Gaussian noise to a named
continuous feature (cell size) and requires the reserved 'unassigned'
cell-type slot. This cohort keeps `cell_size` in `obs`, not in the marker
matrix, so we use `uniform`, which drops edges and features independently of
structure and has no feature-name dependency.


In [ ]:
run(
    [
        "grass_mil.train",
        "task=pretrain_bgrl",
        "model=bgrl_module",
        "trainer=cpu",
        "+augmentation@task.augmentation=uniform",
        *DATA_ARGS,
        f"paths.log_dir={WORK}/logs",
        "trainer.max_epochs=10",
    ],
    "BGRL pretraining",
)

bgrl_ckpt = sorted(
    glob.glob(f"{WORK}/logs/train/runs/*/checkpoints/*.ckpt"), key=os.path.getmtime
)[-1]
print("pretrained encoder:", bgrl_ckpt)

## 2. Attach a supervised head

`model.init_from_ckpt` loads the pretrained encoder weights into a supervised
module; `encoder_init_map=auto_bgrl_or_identity` handles the BGRL key names.

As flagged above, this run is not a result -- it exists to produce a
checkpoint `grass-mil-predict` can consume.


In [ ]:
run(
    [
        "grass_mil.train",
        "task=finetune_mean",
        "model=supervised_module",
        "trainer=cpu",
        f"model.init_from_ckpt={bgrl_ckpt}",
        "model.encoder_init_map=auto_bgrl_or_identity",
        *DATA_ARGS,
        f"data.graph_labels.label_file={RAW}/labels.csv",
        "data.graph_labels.id_column=region_id",
        "data.graph_labels.tasks=[donor_label]",
        "data.graph_labels.scope=region",
        f"paths.log_dir={WORK}/logs",
        "trainer.max_epochs=1",
    ],
    "supervised head",
)

ft_ckpt = sorted(glob.glob(f"{WORK}/logs/train/runs/*/checkpoints/*.ckpt"), key=os.path.getmtime)[
    -1
]
print("checkpoint:", ft_ckpt)

## 3. Export per-instance embeddings

`grass-mil-predict` runs inference on the held-out split and writes the tables
the interpretability suite consumes. Enabling the ShaDow sampler means each
*instance* is an ego-graph around one cell, so the exported table describes
local neighbourhoods rather than whole patches.


In [ ]:
run(
    [
        "grass_mil.inference.predict",
        f"ckpt_path={ft_ckpt}",
        "task=finetune_mean",
        "model=supervised_module",
        "trainer=cpu",
        *DATA_ARGS,
        "data.sampler.name=shadow_custom",
        "data.sampler.runtime.enabled=true",
        "data.sampler.runtime.depth=2",
        "data.sampler.runtime.num_neighbors=8",
        f"data.graph_labels.label_file={RAW}/labels.csv",
        "data.graph_labels.id_column=region_id",
        "data.graph_labels.tasks=[donor_label]",
        "data.graph_labels.scope=region",
        "interpretability.enabled=true",
        "embeddings.enabled=true",
        f"paths.log_dir={WORK}/logs",
    ],
    "predict",
)

instance_table = sorted(
    glob.glob(f"{WORK}/logs/predict/runs/*/predictions/instance_table.csv"), key=os.path.getmtime
)[-1]
inst = pd.read_csv(instance_table)
print(instance_table)
print("instances:", len(inst))
print("embedding dims:", sum(c.startswith("inst_emb_") for c in inst.columns))
print("composition   :", [c for c in inst.columns if c.startswith("comp_")])

> **Known limitation.** `interpretability.spatial.enabled=true` would also
> export a `spatial_table` of edges, which the Moran's I and Ripley plugins
> need. It currently fails when the ShaDow sampler is active: the sampler
> builds batches without `Batch.from_data_list()`, so `to_data_list()` raises
> and the per-instance subgraphs are never collected. The niche analysis below
> does not depend on it.


## 4. Discover niches

The report reduces the embedding space, clusters it into niches, orders them
by composition, and renders an HTML document. Niches are named in dendrogram
order; unassigned instances are labelled `Background`.


In [ ]:
run(
    [
        "grass_mil.interpretability.report_cli",
        f"data.instance_table={instance_table}",
        f"paths.log_dir={WORK}/logs",
        # report.yaml gets its own hydra run dir; pin it so outputs land
        # in the work directory instead of ./outputs/<date>/<time>/
        f"hydra.run.dir={WORK}/report",
        "report.pdf.enabled=false",
    ],
    "interpretability report",
)

report = sorted(glob.glob(f"{WORK}/report/**/report.html", recursive=True), key=os.path.getmtime)[
    -1
]
labels = sorted(
    glob.glob(f"{WORK}/report/**/artifacts/niche_labels.csv", recursive=True), key=os.path.getmtime
)[-1]
print("report:", report)
niches = pd.read_csv(labels)
print(niches["niche_label"].value_counts().sort_index().to_string())

Open `report.html` in a browser for the full document. The rest of this
vignette pulls the same numbers out directly.


## 5. What distinguishes each niche?

Niches are clusters in embedding space; their meaning comes from the cell-type
composition of the neighbourhoods assigned to them.


In [ ]:
merged = inst.merge(niches, on="instance_id")
comp_cols = [c for c in merged.columns if c.startswith("comp_")]

profile = merged.groupby("niche_label")[comp_cols].mean()
profile.columns = [c.replace("comp_", "") for c in profile.columns]
profile.round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(profile.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(profile.shape[1]))
ax.set_xticklabels(profile.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(profile.shape[0]))
ax.set_yticklabels([f"niche {i}" if i != -1 else "Background" for i in profile.index], fontsize=8)
ax.set_title("mean cell-type composition per niche")
fig.colorbar(im, ax=ax, label="fraction")
plt.tight_layout()

## 6. Where are the niches in the tissue?

The instance table carries each ego-graph's centroid, so niches can be mapped
straight back onto the image.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for lab, grp in merged.groupby("niche_label"):
    name = "Background" if lab == -1 else f"niche {lab}"
    ax.scatter(
        grp["center_x"],
        grp["center_y"],
        s=10,
        label=name,
        alpha=0.85,
        color="0.75" if lab == -1 else None,
    )
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
ax.set_title("niches in situ (held-out image)")
ax.set_aspect("equal")
ax.legend(fontsize=8)
plt.tight_layout()

## Where to go next

- **A real cohort.** With enough samples for a genuine sample-level split,
  swap `task=finetune_mean` for `task=finetune_mil` to get per-class attention
  and the margin attribution described in `docs/method_fidelity.md`.
- **Tuning the niches.** `clustering` and `reduction` are config groups. Note
  that `report.yaml` is `@package _global_`, so switching a preset needs the
  package form: `interpretability/clustering@clustering=kmeans`.
- **More analyses.** `interpretability/plugins@plugins=full` enables
  neighbourhood enrichment, filtration curves and the spatial statistics.

See `docs/workflows_train_infer.md` for the full set of workflows and
`docs/hyperparameter_reference.md` for every configuration key.
